# NEX-GDDP-CMIP6 Dataset Ingestion

This notebook creates virtual Zarr references from NEX-GDDP-CMIP6 NetCDF files on S3 and registers them as GeoEngine datasets.

## Prerequisites

- Python 3.10+
- Required packages: `s3fs`, `fsspec`, `kerchunk`, `requests`, `geoengine`, `rasterio` (only for the optional read-back check)
- Access to the NEX-GDDP-CMIP6 S3 bucket (public, no credentials required)
- GeoEngine instance with dataset creation permissions

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import fsspec
import s3fs
from kerchunk.hdf import SingleHdf5ToZarr
from kerchunk.combine import MultiZarrToZarr

import geoengine as ge
import geoengine_api_client as geoc

## Configuration

Update these paths to match your environment.

In [ ]:
# Unused unless the volume type requires an upload flow (no filesystem path);
# keep as the local staging dir for that case.
REFERENCE_PATH = Path("/data/nexgddp-cmip6-refs")

# NEX-GDDP-CMIP6 S3 bucket (public)
S3_BUCKET = "nex-gddp-cmip6"
S3_PREFIX = "NEX-GDDP-CMIP6"

# GeoEngine connection
GEOENGINE_URL = "http://localhost:3030/api"
GEOENGINE_USER = "admin@localhost"
GEOENGINE_PASSWORD = "adminadmin"

# Volume name in GeoEngine
VOLUME_NAME = "ge_geo_data"

# Backend model registry (single source of truth; keep MODELS below in sync)
REGISTRY_PATH = Path("../backend/conf/nexgddp_models.toml")

# Starter models (5 models as per plan)
MODELS = [
    {"id": "ACCESS-CM2", "variant": "r1i1p1f1", "grid": "gn"},
    {"id": "MPI-ESM1-2-LR", "variant": "r1i1p1f1", "grid": "gn"},
    {"id": "IPSL-CM6A-LR", "variant": "r1i1p1f1", "grid": "gn"},
    {"id": "MRI-ESM2-0", "variant": "r1i1p1f1", "grid": "gn"},
    {"id": "GFDL-ESM4", "variant": "r1i1p1f1", "grid": "gr1"},
]

# Scenarios
SCENARIOS = ["historical", "ssp245", "ssp585"]

# Variables
VARIABLES = ["tasmax", "tasmin", "pr"]

# Variable metadata
VARIABLE_CONFIGS = {
    "tasmax": {
        "name": "Maximum Temperature",
        "measurement": "temperature",
        "unit": "K",
        "min": 200,
        "max": 340,
        "color_map": "coolwarm",
    },
    "tasmin": {
        "name": "Minimum Temperature",
        "measurement": "temperature",
        "unit": "K",
        "min": 200,
        "max": 340,
        "color_map": "coolwarm",
    },
    "pr": {
        "name": "Precipitation",
        "measurement": "precipitation",
        "unit": "kg m-2 s-1",
        "min": 0,
        "max": 0.01,
        "color_map": "Blues",
    },
}

In [ ]:
try:
    import tomllib
except ModuleNotFoundError:  # Python < 3.11
    import tomli as tomllib

with open(REGISTRY_PATH, "rb") as fh:
    registry = tomllib.load(fh)["model"]

registry_ids = {m["id"] for m in registry}
notebook_ids = {m["id"] for m in MODELS}
missing = notebook_ids - registry_ids
extra = registry_ids - notebook_ids
assert not missing, f"MODELS not in backend registry: {missing}"
if extra:
    print(f"Warning: backend registry has models not in MODELS: {extra}")

## Initialize GeoEngine Connection

In [ ]:
ge.initialize(
    GEOENGINE_URL,
    credentials=(GEOENGINE_USER, GEOENGINE_PASSWORD),
)

volume = ge.volume_by_name(VOLUME_NAME)
print(f"Using volume: {VOLUME_NAME} (path={volume.path!r})")

# Reference files must live where GeoEngine can reach them. If the volume has a
# filesystem path, write refs under it; otherwise an upload step is required.
DATA_DIR = Path(volume.path) / "nexgddp-cmip6" if volume.path else None
if DATA_DIR is None:
    print("Warning: volume has no filesystem path - reference files must be uploaded")

permission_tuples = [
    (ge.permissions.ANONYMOUS_USER_ROLE_ID, ge.Permission.READ),
    (ge.permissions.REGISTERED_USER_ROLE_ID, ge.Permission.READ),
]

## Create Reference Files

For each model/scenario/variable combination, create a kerchunk reference file that combines all yearly NetCDF files into a virtual Zarr store.

In [ ]:
def list_netcdf_files(model_id, scenario, variable, variant, grid):
    """List all yearly NetCDF files for a model/scenario/variable combination."""
    fs = s3fs.S3FileSystem(anon=True)
    
    # Determine year range based on scenario
    if scenario == "historical":
        years = range(1950, 2015)
    else:
        years = range(2015, 2101)
    
    files = []
    for year in years:
        filename = f"{variable}_day_{model_id}_{scenario}_{variant}_{grid}_{year}.nc"
        path = f"{S3_BUCKET}/{S3_PREFIX}/{model_id}/{scenario}/{variant}/{variable}/{filename}"
        if fs.exists(path):
            files.append(path)
        else:
            print(f"Warning: File not found: {path}")
    
    return sorted(files)

In [ ]:
def create_kerchunk_reference(files, output_path):
    """Create a kerchunk reference file from a list of NetCDF files.

    Returns the combined reference dict (so metadata like the time
    range and grid can be derived during registration).
    """
    fs = s3fs.S3FileSystem(anon=True)
    
    # Create individual references for each file
    individual_refs = []
    for file_path in files:
        with fs.open(file_path, "rb") as f:
            h5chunks = SingleHdf5ToZarr(f, file_path, inline_threshold=100)
            individual_refs.append(h5chunks.translate())
    
    # Combine references along time dimension
    mzz = MultiZarrToZarr(
        individual_refs,
        concat_dims=["time"],
        identical_dims=["lat", "lon"],
    )
    combined = mzz.translate()
    
    # Write reference file
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        json.dump(combined, f)
    
    print(f"Created reference: {output_path}")
    return combined

In [ ]:
# Create reference files for all combinations (staging dir as fallback)
assert DATA_DIR is not None, \
    "No volume path - place refs via upload or mount the volume before running"
OUTPUT_DIR = DATA_DIR

for model in MODELS:
    for scenario in SCENARIOS:
        for variable in VARIABLES:
            print(f"\nProcessing: {model['id']} / {scenario} / {variable}")
            
            files = list_netcdf_files(
                model["id"],
                scenario,
                variable,
                model["variant"],
                model["grid"],
            )
            
            if not files:
                print(f"No files found, skipping")
                continue
            
            output_path = OUTPUT_DIR / model["id"] / scenario / f"{variable}.json"
            create_kerchunk_reference(files, output_path)

## Register Datasets in GeoEngine

Register each reference file as a GeoEngine dataset.

In [ ]:
def build_netcdf_cf_metadata(variable, scenario, year_start, year_end, reference_path):
    """Build the NetCDF-CF metadata a kerchunk reference needs for registration.

    Targets geoengine-api-client 0.9.4 model shapes. RUNTIME-VERIFY against the
    running instance before a full run:
    - that GDAL opens the reference as NetCDF-CF with one raster band per day;
    - the exact gdal_config_options (`GDAL_ZARR_VFS`?) for reading kerchunk refs;
    - the 0.25 degree grid bounds against the generated reference.
    """
    config = VARIABLE_CONFIGS[variable]

    # NEX-GDDP-CMIP6: 0.25 degree global grid on EPSG:4326
    n_lat, n_lon, dx = 721, 1440, 0.25
    x_min, y_max = 0.0, 90.0

    measurement = geoc.ContinuousMeasurement(
        measurement=config["unit"], unit=config["unit"]
    )
    result_descriptor = geoc.RasterResultDescriptor(
        data_type="F32",
        spatial_reference="EPSG:4326",
        measurement=measurement,
        bands=[geoc.RasterBandDescriptor(name=variable, measurement=measurement)],
    )

    start_ms = int(datetime(year_start, 1, 1).timestamp() * 1000)
    end_ms = int(datetime(year_end, 1, 1).timestamp() * 1000)  # non-inclusive
    params = geoc.GdalDatasetParameters(
        file_path=str(reference_path),
        rasterband_channel=1,
        geo_transform=geoc.GeoTransform(
            origin_coordinate=geoc.Coordinate2D(x=x_min, y=y_max),
            x_pixel_size=dx,
            y_pixel_size=-dx,
        ),
        width=n_lon,
        height=n_lat,
        file_not_found_handling="Abort",
        gdal_config_options=[["GDAL_ZARR_VFS", f"REF:{reference_path}"]],
    )

    return geoc.GdalMetadataNetCdfCf(
        type="GdalMetaDataNetCdfCf",
        result_descriptor=result_descriptor,
        params=params,
        start=start_ms,
        end=end_ms,
        step=geoc.TimeStep(granularity="Milliseconds", step=86400000),
        band_offset=0,
    )

def register_dataset(model_id, scenario, variable, reference_path, year_start, year_end):
    """Register a dataset in GeoEngine using the volume add-or-replace API."""
    config = VARIABLE_CONFIGS[variable]
    dataset_name = f"nexgddp_cmip6_{model_id}_{scenario}_{variable}"
    display_name = f"NEX-GDDP-CMIP6 {config['name']} ({scenario})"

    properties = ge.AddDatasetProperties(
        name=dataset_name,
        display_name=display_name,
        description=(
            f"Daily NEX-GDDP-CMIP6 {config['name']} projections for {config['measurement']} ({config['unit']}). "
            f"Model: {model_id}, Scenario: {scenario}"
        ),
        source_operator="GdalSource",
        symbology=ge.RasterSymbology(
            raster_colorizer=ge.SingleBandRasterColorizer(
                band=0,
                band_colorizer=ge.Colorizer.linear_with_mpl_cmap(
                    color_map=config["color_map"],
                    min_max=(config["min"], config["max"]),
                    n_steps=10,
                ),
            ),
        ),
    )

    meta_data = geoc.MetaDataDefinition(
        build_netcdf_cf_metadata(variable, scenario, year_start, year_end, reference_path)
    )

    volume.add_or_replace_dataset_with_permissions(
        volume,
        properties,
        meta_data,
        permission_tuples=permission_tuples,
        replace_existing=True,
        timeout=600,
    )
    print(f"Registered: {dataset_name}")
    return dataset_name

In [ ]:
# Register all datasets
YEAR_RANGES = {"historical": (1950, 2015), "ssp245": (2015, 2101), "ssp585": (2015, 2101)}
registered_datasets = []

for model in MODELS:
    for scenario in SCENARIOS:
        year_start, year_end = YEAR_RANGES[scenario]
        for variable in VARIABLES:
            rel_ref = Path("nexgddp-cmip6") / model["id"] / scenario / f"{variable}.json"
            ref_abs = DATA_DIR / rel_ref
            if ref_abs.exists():
                dataset_name = register_dataset(
                    model["id"],
                    scenario,
                    variable,
                    rel_ref,  # path is relative to the volume root
                    year_start,
                    year_end,
                )
                registered_datasets.append(dataset_name)

print(f"\nTotal datasets: {len(registered_datasets)}")

## Verification

Verify that GDAL can read the reference files.

In [ ]:
import rasterio

test_ref = DATA_DIR / MODELS[0]["id"] / SCENARIOS[0] / f"{VARIABLES[0]}.json"
if test_ref.exists():
    try:
        # GDAL's Zarr driver needs configuration to read kerchunk references
        # (e.g. the vfs/REF mechanism); adjust per your GDAL build.
        with rasterio.open(f"zarr://{test_ref}") as src:
            print(f"Successfully opened reference file")
            print(f"  CRS: {src.crs}")
            print(f"  Bounds: {src.bounds}")
            print(f"  Shape: {src.shape}")
    except Exception as e:
        print(f"Failed to open reference file: {e}")
        print("Note: GDAL may need additional configuration to read kerchunk references.")
else:
    print(f"Test reference file not found: {test_ref}")

# Confirm GeoEngine sees the registered dataset
for ds in registered_datasets[:3]:
    info = ge.dataset_info_by_name(ds)
    print(f"{ds}: {info is not None}")

## Next Steps

1. Verify that GeoEngine can read the kerchunk reference files
2. Update the `register_dataset` function with proper metadata extraction
3. Run the full registration to create all 45 datasets
4. Test the climate-risk process with the new datasets